# 00 · Colab Setup (run this first)

Bootstrap on Google Colab: get the repo, install deps, add the Kaggle token, download the **pkdarabi/cardetection** dataset, and build the YOLO + COCO data.

**Before running:** Runtime → Change runtime type → **Hardware accelerator: GPU**.

## 1. Get the repo into Colab
Edit `REPO_URL` to your fork, then run.

In [ ]:
REPO_URL = 'https://github.com/<you>/traffic-sign-detection-yolo-detr.git'
import os
if not os.path.isdir('/content/traffic-sign-detection-yolo-detr'):
    !git clone $REPO_URL /content/traffic-sign-detection-yolo-detr
%cd /content/traffic-sign-detection-yolo-detr
!ls

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt
import torch; print('CUDA:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 3. Add your Kaggle token
Upload the `kaggle.json` from kaggle.com → Settings → API → Create New Token.

In [ ]:
from google.colab import files
import os, shutil
print('Select your kaggle.json ...')
up = files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('kaggle.json installed')

## 4. Download the teacher's dataset (pkdarabi/cardetection)

In [ ]:
!kaggle datasets download -d pkdarabi/cardetection -p data/raw/cardetection --unzip
!find data/raw/cardetection -maxdepth 2 | head -40

### Inspect the Roboflow data.yaml (confirm the 15 class names + order)

In [ ]:
import glob
for y in glob.glob('data/raw/cardetection/**/data.yaml', recursive=True)[:1]:
    print(open(y).read())

## 5. Build the dataset: normalize layout + sync configs + COCO for DETR

In [ ]:
# --write-configs regenerates configs/{classes,cardetection}.yaml from the raw data.yaml
!python -m src.data.prepare_dataset --write-configs
!python -m src.data.convert_to_coco
!python -m src.data.visualize_annotations --yolo-root data/processed/yolo/cardetection --split val --n 12

In [ ]:
from IPython.display import Image
Image('results/plots/annotation_check.png')

## Next
- **01_eda** — dataset analysis (midterm deliverable)
- **02_yolo_baseline** / **03_detr_baseline**
- **04_comparison**, **05_augmentation_ablation**, **06_limited_data**, **07_robustness**, **08_error_analysis**